## Enhancement + Segmentation

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

### Enhancement Part

In [ ]:
# Create Output Folders for Enhanced Images
input_folder = "../frames/selected"
output_folder = "../frames/enhanced"

folders = ["1_Grayscale", "2_NoiseReduced", "3_IlluminationCorrected","4_FinalEnhanced"]

# Create main folder
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Create subfolders
for folder in folders:
    path = os.path.join(output_folder, folder)
    if not os.path.exists(path):
        os.makedirs(path)

print("All folders created!")

In [ ]:
# Convert images to grayscale and save in the folder

# Loop through all selected images
for file in os.listdir(input_folder):
    if file.endswith((".jpg", ".png", ".jpeg")):
        img = cv2.imread(os.path.join(input_folder, file))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        cv2.imwrite(os.path.join(output_folder, "1_Grayscale", file), gray)

print("Grayscale conversion completed!")

In [ ]:
# Noise Reduction + Enhancement

frames_folder = "../frames/enhanced/1_Grayscale"
output_folder = "../frames/enhanced/2_NoiseReduced"

os.makedirs(output_folder, exist_ok=True)

# Process only few images for analysis
max_images = 3
count = 0

for file in os.listdir(frames_folder):

    if not file.endswith((".jpg", ".png", ".jpeg")):
        continue

    path = os.path.join(frames_folder, file)

    img = cv2.imread(path)

    if img is None:
        continue

    # ---------------- STEP 1 : MEDIAN FILTER ----------------
    # Removes small asphalt noise while preserving crack lines
    median = cv2.medianBlur(img, 3)

    # ---------------- STEP 2 : GAUSSIAN FILTER ----------------
    # Smooths image noise
    denoised = cv2.GaussianBlur(median, (5,5), 0)

    # ---------------- STEP 3 : CLAHE CONTRAST ENHANCEMENT ----------------
    # Improves local crack visibility
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray = cv2.cvtColor(denoised, cv2.COLOR_BGR2GRAY)
    enhanced = clahe.apply(gray)

    # ---------------- STEP 4 : LIGHT SHARPENING ----------------
    # Makes crack edges clearer for segmentation
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]])

    sharpened = cv2.filter2D(enhanced, -1, kernel)

    # Save final enhanced image
    cv2.imwrite(os.path.join(output_folder, file), sharpened)

    # ---------------- VISUAL ANALYSIS ----------------
    if count < max_images:

        plt.figure(figsize=(15,5))

        plt.subplot(1,3,1)
        plt.imshow(img, cmap="gray")
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1,3,2)
        plt.imshow(denoised, cmap="gray")
        plt.title("Noise Reduced")
        plt.axis("off")

        plt.subplot(1,3,3)
        plt.imshow(sharpened, cmap="gray")
        plt.title("Enhanced Final")
        plt.axis("off")

        plt.suptitle(f"Noise Reduction + Enhancement: {file}")
        plt.show()

        # Histogram Comparison
        plt.figure(figsize=(7,4))

        hist1 = cv2.calcHist([img],[0],None,[256],[0,256])
        hist2 = cv2.calcHist([sharpened],[0],None,[256],[0,256])

        plt.plot(hist1, label="Original")
        plt.plot(hist2, label="Enhanced")

        plt.title("Histogram Comparison")
        plt.legend()
        plt.show()

        count += 1

print("Noise reduction and enhancement completed.")

In [ ]:
# Illumination Correction

input_folder = "../frames/enhanced/2_NoiseReduced"
output_folder = "../frames/enhanced/3_IlluminationCorrected"

os.makedirs(output_folder, exist_ok=True)

max_images = 3
count = 0

# Contrast Stretching function
def contrast_stretch(img):

    min_val = np.min(img)
    max_val = np.max(img)

    # Avoid division crash
    if max_val - min_val == 0:
        return img.copy()

    stretched = (img - min_val) * (255 / (max_val - min_val))

    return np.clip(stretched, 0, 255).astype(np.uint8)


for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # ---------------- CONTRAST STRETCHING ---------------- #
        contrast_img = contrast_stretch(img)

        # ---------------- CLAHE (BETTER THAN HISTOGRAM EQUALIZATION) ---------------- #
        # Prevents over-enhancement of road texture

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8,8)
        )

        illum_corrected = clahe.apply(contrast_img)

        # ---------------- LIGHT SMOOTHING ---------------- #
        # Small smoothing to reduce enhanced texture noise

        illum_corrected = cv2.GaussianBlur(
            illum_corrected,
            (3,3),
            0
        )

        # Save final image
        cv2.imwrite(
            os.path.join(output_folder, file),
            illum_corrected
        )

        # ---------------- VISUAL ANALYSIS ---------------- #

        if count < max_images:

            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.imshow(img, cmap="gray")
            plt.title("Original")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(contrast_img, cmap="gray")
            plt.title("Contrast Stretching")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(illum_corrected, cmap="gray")
            plt.title("CLAHE Corrected")
            plt.axis("off")

            plt.suptitle(f"Illumination Correction: {file}")
            plt.show()

            # Histogram Comparison
            plt.figure(figsize=(8,4))

            hist1 = cv2.calcHist([img],[0],None,[256],[0,256])
            hist2 = cv2.calcHist([illum_corrected],[0],None,[256],[0,256])

            plt.plot(hist1, label="Original")
            plt.plot(hist2, label="Corrected")

            plt.title("Histogram Improvement")
            plt.xlabel("Pixel Intensity")
            plt.ylabel("Frequency")

            plt.legend()
            plt.show()

            count += 1

print("Illumination correction completed!")

In [ ]:
# Motion blur and camera instability analysis

input_folder = "../frames/enhanced/3_IlluminationCorrected"
output_folder = "../frames/enhanced/4_FinalEnhanced"

os.makedirs(output_folder, exist_ok=True)

# ---------------- CREATE CATEGORY FOLDERS ---------------- #

categories = [
    "AligatorCracking",
    "TransverseCracking",
    "Pothole",
    "Raveling"
]

for category in categories:

    os.makedirs(
        os.path.join(output_folder, category),
        exist_ok=True
    )

max_images = 3
count = 0


# Sharpness (Laplacian)
def calculate_sharpness(img):

    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return cv2.Laplacian(img, cv2.CV_64F).var()


# Edge Strength (Sobel)
def calculate_edge_strength(img):

    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(sobelx**2 + sobely**2)

    return np.mean(magnitude)


# Blur Detection
def detect_blur_type(sharpness, edge_strength):

    # Slightly adjusted thresholds for road images

    if sharpness < 50:
        return "Strong Blur"

    elif sharpness < 100:
        return "Moderate Blur"

    elif edge_strength < 20:
        return "Edge Loss"

    else:
        return "Sharp"


# Enhancement Methods
def enhance_image(img, blur_type):

    # ---------------- LIGHT EDGE-PRESERVING SMOOTHING ---------------- #
    # Better than strong Gaussian blur for crack preservation

    base = cv2.GaussianBlur(
        img,
        (3,3),
        0
    )

    # ---------------- STRONG BLUR ---------------- #

    if blur_type == "Strong Blur":

        # Gentle unsharp masking
        gaussian = cv2.GaussianBlur(base, (0,0), 2.0)

        enhanced = cv2.addWeighted(
            base,
            1.4,
            gaussian,
            -0.4,
            0
        )

    # ---------------- MODERATE BLUR ---------------- #

    elif blur_type == "Moderate Blur":

        gaussian = cv2.GaussianBlur(base, (0,0), 1.5)

        enhanced = cv2.addWeighted(
            base,
            1.3,
            gaussian,
            -0.3,
            0
        )

    # ---------------- EDGE LOSS ---------------- #

    elif blur_type == "Edge Loss":

        # Mild sharpening only
        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])

        enhanced = cv2.filter2D(base, -1, kernel)

    # ---------------- SHARP ---------------- #

    else:

        # Keep natural appearance
        enhanced = cv2.convertScaleAbs(
            base,
            alpha=1.05,
            beta=0
        )

    # ---------------- FINAL SAFETY SMOOTHING ---------------- #
    # Prevents sharpening artifacts

    enhanced = cv2.GaussianBlur(enhanced, (3,3), 0)

    return enhanced


# Get images
image_files = []

for file in os.listdir(input_folder):

    if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        image_files.append(file)


# Process each image
for filename in image_files:

    path = os.path.join(input_folder, filename)

    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        print("Failed to load:", filename)
        continue

    # Before
    sharp_before = calculate_sharpness(image)
    edge_before = calculate_edge_strength(image)

    # Classify
    blur_type = detect_blur_type(
        sharp_before,
        edge_before
    )

    # Enhance
    enhanced = enhance_image(image, blur_type)

    # After
    sharp_after = calculate_sharpness(enhanced)
    edge_after = calculate_edge_strength(enhanced)

    # ---------------- SELECT CATEGORY FOLDER ---------------- #

    if "aligator" in filename.lower():

        save_folder = "AligatorCracking"

    elif "transverse" in filename.lower():

        save_folder = "TransverseCracking"

    elif "pothole" in filename.lower():

        save_folder = "Pothole"

    elif "raveling" in filename.lower():

        save_folder = "Raveling"

    else:

        save_folder = ""

    # Save
    save_path = os.path.join(
        output_folder,
        save_folder,
        "enhanced_" + filename
    )

    cv2.imwrite(save_path, enhanced)

    # Visualization
    if count < max_images:

        print("\n==============================")
        print("Processing:", filename)

        print("Before Sharpness:", sharp_before)
        print("Before Edge Strength:", edge_before)

        print("Detected Type:", blur_type)

        print("After Sharpness:", sharp_after)
        print("After Edge Strength:", edge_after)

        edges_original = cv2.Canny(image, 50, 150)
        edges_enhanced = cv2.Canny(enhanced, 50, 150)

        plt.figure(figsize=(18, 6))

        plt.subplot(1, 3, 1)
        plt.imshow(image, cmap='gray')
        plt.title(f"Original\nSharp: {sharp_before:.2f}")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(enhanced, cmap='gray')
        plt.title(f"Enhanced\nSharp: {sharp_after:.2f}")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(edges_enhanced, cmap='gray')
        plt.title("Edge Map (After)")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        count += 1

print("\nAll images processed successfully!")

## Segmentation Part

In [ ]:
# Alligator cracking segmentation

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/AligatorCracking"
output_folder = "../../frames/segmented/AlligatorCracking"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Segment alligator cracks
def segment_alligator(gray):

    # Step 1: Light smoothing
    blur = cv2.GaussianBlur(gray, (3,3), 0)

    # Step 2: Blackhat enhancement
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    # Step 3: Binary threshold
    _, binary = cv2.threshold(
        blackhat,
        20,
        255,
        cv2.THRESH_BINARY
    )

    # Step 4: Remove tiny noise
    open_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3,3)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        open_kernel,
        iterations=1
    )

    # Step 5: Moderate connection
    # Prevent whole-image merging
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        close_kernel,
        iterations=1
    )

    return binary


# Merge nearby boxes
def merge_boxes(boxes, distance=35):

    merged_boxes = []

    while boxes:

        x1, y1, x2, y2 = boxes.pop(0)

        merged = True

        while merged:

            merged = False

            remove_indices = []

            for i, (xx1, yy1, xx2, yy2) in enumerate(boxes):

                # Check nearby distance
                if (
                    abs(xx1 - x2) < distance or
                    abs(x1 - xx2) < distance
                ) and (
                    abs(yy1 - y2) < distance or
                    abs(y1 - yy2) < distance
                ):

                    x1 = min(x1, xx1)
                    y1 = min(y1, yy1)
                    x2 = max(x2, xx2)
                    y2 = max(y2, yy2)

                    remove_indices.append(i)

                    merged = True

            for index in sorted(remove_indices, reverse=True):
                boxes.pop(index)

        merged_boxes.append([x1, y1, x2, y2])

    return merged_boxes


# Detect alligator crack regions
def detect_alligator_regions(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    # Small crack cell detection
    for c in contours:

        area = cv2.contourArea(c)

        if 100 < area < 2500:

            x, y, w, h = cv2.boundingRect(c)

            aspect_ratio = w / float(h + 1e-5)

            # Ignore long lines
            if 0.4 < aspect_ratio < 3.5:

                boxes.append([x, y, x+w, y+h])

    # Merge nearby boxes moderately
    merged_boxes = merge_boxes(boxes)

    # Draw medium-sized regions only
    for box in merged_boxes:

        x1, y1, x2, y2 = box

        width = x2 - x1
        height = y2 - y1

        area = width * height

        # Prevent huge full-image box
        if 3000 < area < 60000:

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                3
            )

            cv2.putText(
                output,
                "Alligator Crack",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Segment cracks
        mask = segment_alligator(gray)

        # Detect regions
        result = detect_alligator_regions(img, mask)

        # Save
        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(gray, cmap="gray")
            plt.title("Enhanced Input")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(mask, cmap="gray")
            plt.title("Crack Segmentation")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Alligator Crack Detection")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Alligator crack detection completed successfully!")

In [ ]:
# Pothole detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/Pothole"
output_folder = "../../frames/segmented/Pothole"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Pothole detection function
def detect_pothole(gray, output, mask):

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        95,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.bitwise_and(th, th, mask=mask)

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_OPEN,
        np.ones((9,9), np.uint8)
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        np.ones((15,15), np.uint8)
    )

    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 1500:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w / (h + 1e-5) > 3.5:
            continue

        cv2.rectangle(
            output,
            (x,y),
            (x+w,y+h),
            (0,0,255),
            2
        )

        cv2.putText(
            output,
            "Pothole",
            (x,y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

        count += 1

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)
        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Dummy full mask (since your original code expects mask)
        mask = np.ones_like(gray, dtype=np.uint8) * 255

        output = img.copy()

        # Step: pothole detection
        result = detect_pothole(gray, output, mask)

        # Save output
        cv2.imwrite(os.path.join(output_folder, file), result)

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15, 5))

            plt.subplot(1, 3, 1)
            plt.imshow(gray, cmap="gray")
            plt.title("Input")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(mask, cmap="gray")
            plt.title("Mask")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Pothole Detection")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Pothole detection completed successfully!")

In [ ]:
# Raveling detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/Raveling"
output_folder = "../../frames/segmented/Raveling"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Raveling detection function
def apply_enhancement(img):

    # Step 1: Smooth noise
    smoothed = cv2.blur(img, (3, 3))

    # Step 2: Logarithmic transformation
    img_float = smoothed.astype(np.float32)

    constant = 100 / np.log(
        1 + np.max(img_float)
    )

    log_img = constant * np.log(
        1 + img_float
    )

    log_img = np.array(
        log_img,
        dtype=np.uint8
    )

    # Step 3: Contrast Stretching
    low = np.min(log_img)
    high = np.max(log_img)

    if high <= low:
        return log_img

    stretched = cv2.convertScaleAbs(
        log_img,
        alpha=(255.0 / (high - low)),
        beta=-(low * 255.0 / (high - low))
    )

    return stretched

# Extract raveling damage
def extract_damage(enhanced_img):

    gaussian = cv2.GaussianBlur(
        enhanced_img,
        (5, 5),
        0
    )

    # Binarization
    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15,
        3
    )

    # Noise removal
    struct_element = np.ones(
        (3, 3),
        np.uint8
    )

    refined_mask = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        struct_element,
        iterations=2
    )

    return refined_mask


# Detect raveling regions
def detect_raveling(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        if cv2.contourArea(c) > 150:

            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                output,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                output,
                "Raveling",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        # Apply enhancement
        enhanced = apply_enhancement(gray)

        # Extract damage
        mask = extract_damage(enhanced)

        # Detect raveling
        result = detect_raveling(img, mask)

        # Save result
        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(gray, cmap="gray")
            plt.title("Enhanced Input")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(mask, cmap="gray")
            plt.title("Raveling Mask")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Detected Raveling")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Raveling detection completed successfully!")

In [ ]:
# Transverse cracking detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/TransverseCracking"
output_folder = "../../frames/segmented/TransverseCracking"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0

# Transverse extraction function
def extract_transverse(gray):

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.5,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(gray)

    # Blur
    blur = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Dark cracks become bright
    blackhat_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (21, 21)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        blackhat_kernel
    )

    # Treshold
    _, binary = cv2.threshold(
        blackhat,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Remove small noise
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8),
        iterations=1
    )

    # Horizontal extraction 
    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (45, 5)
    )

    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        horizontal_kernel
    )

    # Merge nearby horizontal parts
    merge_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (35, 7)
    )

    merged = cv2.dilate(
        horizontal,
        merge_kernel,
        iterations=1
    )

    merged = cv2.morphologyEx(
        merged,
        cv2.MORPH_CLOSE,
        np.ones((9, 9), np.uint8),
        iterations=1
    )

    return merged

# Detect transverse crack regions
def detect_transverse(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    H, W = image.shape[:2]

    for c in contours:

        area = cv2.contourArea(c)

        # Remove small regions
        if area < 1000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w < 80:
            continue

        # Aspect ratio
        aspect_ratio = w / (h + 1e-5)

        if aspect_ratio < 2.5:
            continue

        # Draw detected transverse crack
        cv2.rectangle(
            output,
            (x, y),
            (x + w, y + h),
            (0, 255, 255),
            3
        )

        cv2.putText(
            output,
            "Transverse Crack",
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2,
            cv2.LINE_AA
        )

    return output


# Main processing loop
if not os.path.isdir(input_folder):
    print(f"Input folder does not exist: {input_folder}")
else:
    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".png", ".jpeg")):

            path = os.path.join(input_folder, file)

            img = cv2.imread(path)

            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            mask = extract_transverse(gray)

            result = detect_transverse(img, mask)

            # Save output
            save_path = os.path.join(output_folder, file)
            cv2.imwrite(save_path, result)

            # Visualization
            if count < max_show:

                plt.figure(figsize=(18, 6))

                plt.subplot(1, 3, 1)
                plt.imshow(gray, cmap="gray")
                plt.title("Input")
                plt.axis("off")

                plt.subplot(1, 3, 2)
                plt.imshow(mask, cmap="gray")
                plt.title("Transverse Mask")
                plt.axis("off")

                plt.subplot(1, 3, 3)
                plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
                plt.title("Detected Transverse Cracks")
                plt.axis("off")

                plt.tight_layout()
                plt.show()

                count += 1

print("Transverse crack detection completed!")